In [ ]:
import pickle
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import classification_report

# Charger les données de test

    
with open('x_test.pkl', 'rb') as file:
    content = file.read(100)  # Lire les 100 premiers octets
print(content)


with open('y_test.pkl', 'rb') as file:
    y_test = pickle.load(file)

# Charger le modèle
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained('dmis-lab/biobert-base-cased-v1.1', num_labels=... )  # Ajustez num_labels
model.load_state_dict(torch.load('model_biobert.pth', map_location=device))
model.to(device)
model.eval()

# Charger le tokenizer
tokenizer = BertTokenizer.from_pretrained('dmis-lab/biobert-base-cased-v1.1')

# Charger le label encoder (si utilisé)
with open('label_encoder.pkl', 'rb') as file:
    label_encoder = pickle.load(file)


In [ ]:
import pandas as pd

with open('y_test.pkl', 'rb') as file:
    content = file.read(100)  # Lire les 100 premiers octets
print(content)



In [ ]:
import joblib

# Charger y_test.pkl avec joblib
y_test = joblib.load('y_test.pkl')

# Vérifier le type et un aperçu
print(type(y_test))
print(y_test[:5])  # Afficher les 5 premiers éléments si c'est un tableau ou une liste


In [ ]:
# Charger l'encodeur de labels (LabelEncoder) si disponible
import pickle

with open('label_encoder.pkl', 'rb') as file:
    label_encoder = pickle.load(file)

# Décoder les indices vers les noms de maladies
y_test_decoded = label_encoder.inverse_transform(y_test)
print(y_test_decoded[:5])  # Afficher les 5 premiers labels décodés


In [ ]:
import os

print(os.listdir())  # Lister tous les fichiers du répertoire courant



In [ ]:
import pandas as pd

# Charger le dataset
data = pd.read_csv('data/dataset_cleaned.csv')  # Remplacez par le chemin de votre fichier
print(data.head())  # Aperçu des premières lignes du dataset

# Extraire les maladies uniques
diseases = data['Disease'].unique()  # Remplacez 'Disease' par le nom exact de la colonne
print(diseases)  # Vérifiez les maladies extraites


In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle

# Créer et ajuster le LabelEncoder
label_encoder = LabelEncoder()
label_encoder.fit(diseases)

# Sauvegarder l'encodeur dans un fichier Pickle
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)

print("LabelEncoder sauvegardé avec succès.")


In [ ]:
# Charger le fichier label_encoder.pkl
with open('label_encoder.pkl', 'rb') as file:
    label_encoder = pickle.load(file)

# Vérifiez les classes encodées
print(label_encoder.classes_)  # Affiche toutes les maladies encodées


In [ ]:
# Décoder les indices de y_test
y_test_decoded = label_encoder.inverse_transform(y_test)
print(y_test_decoded[:5])  # Afficher les 5 premiers labels décodés


In [ ]:
import joblib

# Charger les données de test depuis x_test.pkl
X_test = joblib.load('x_test.pkl')
print(X_test[:5])  # Afficher un aperçu des données


In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# Charger le dataset complet
data = pd.read_csv('data/dataset_cleaned.csv')

# Séparer les caractéristiques (symptômes) et les cibles (maladies)
X = data['Symptoms']  # Remplacez 'Symptoms' par la colonne correspondante
y = data['Disease']  # Remplacez 'Disease' par la colonne correspondante

# Diviser les données en train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_test.head())  # Aperçu des données de test


In [ ]:
from transformers import BertTokenizer

# Charger le tokenizer pour le modèle utilisé (par exemple, BioBERT)
tokenizer = BertTokenizer.from_pretrained('dmis-lab/biobert-base-cased-v1.1')


In [ ]:
from transformers import BertForSequenceClassification
import torch

# Charger le modèle entraîné
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained('dmis-lab/biobert-base-cased-v1.1', num_labels=len(label_encoder.classes_))
model.load_state_dict(torch.load('model_biobert.pth', map_location=device))
model.to(device)
model.eval()


In [ ]:
y_pred = []

# Parcourir chaque échantillon dans X_test pour faire des prédictions
for sample in X_test:
    tokens = tokenizer(sample, return_tensors='pt', padding=True, truncation=True).to(device)
    output = model(**tokens)
    predicted_label = torch.argmax(output.logits, dim=1).cpu().item()
    y_pred.append(predicted_label)





print(y_pred[:5])  # Afficher les 5 premières prédictions


In [ ]:
MAX_LEN = 128  # Définir une longueur maximale raisonnable
tokens = tokenizer(sample, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN).to(device)

# Décoder les indices en noms de maladies
y_pred_decoded = label_encoder.inverse_transform(y_pred)
print(y_pred_decoded[:5])  # Afficher les 5 premières prédictions décodées

In [ ]:
# Transformer y_test en indices numériques
y_test_numeric = label_encoder.transform(y_test_decoded)  # y_test_decoded contient les maladies en texte
print(y_test_numeric[:5])  # Vérifiez les premiers indices

# Générer le rapport avec les indices numériques
from sklearn.metrics import classification_report

print("Rapport de classification :")
print(classification_report(y_test_numeric, y_pred, target_names=label_encoder.classes_))


In [ ]:
# Transformer y_pred en noms de maladies
y_pred_decoded = label_encoder.inverse_transform(y_pred)
print(y_pred_decoded[:5])  # Vérifiez les 5 premières prédictions

# Générer le rapport avec les noms de maladies
from sklearn.metrics import classification_report

print("Rapport de classification :")
print(classification_report(y_test, y_pred_decoded, target_names=label_encoder.classes_))


In [ ]:
overlap = set(X_train).intersection(set(X_test))
print(f"Nombre d'exemples en commun entre train et test : {len(overlap)}")


In [ ]:
from sklearn.model_selection import train_test_split

# Diviser les données en train et test sans chevauchement
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin

class SklearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def fit(self, X, y):
        pass  # L'entraînement doit être géré séparément

    def predict(self, X):
        y_pred = []
        for sample in X:
            tokens = self.tokenizer(sample, return_tensors='pt', padding=True, truncation=True, max_length=128).to(self.device)
            output = self.model(**tokens)
            predicted_label = torch.argmax(output.logits, dim=1).cpu().item()
            y_pred.append(predicted_label)
        return y_pred

# Créer une instance du wrapper
wrapped_model = SklearnWrapper(model, tokenizer, device)

# Exécuter la validation croisée
from sklearn.model_selection import cross_val_score
scores = cross_val_score(wrapped_model, X, y, cv=5)
print(f"Scores de validation croisée : {scores}")
print(f"Score moyen : {scores.mean():.2f}")


In [ ]:
class SklearnWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, model, tokenizer, device):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device

    def fit(self, X, y):
        pass  # L'entraînement doit être géré séparément

    def predict(self, X):
        y_pred = []
        for sample in X:
            tokens = self.tokenizer(sample, return_tensors='pt', padding=True, truncation=True, max_length=128).to(self.device)
            output = self.model(**tokens)
            predicted_label = torch.argmax(output.logits, dim=1).cpu().item()
            y_pred.append(predicted_label)

        print(f"Prédictions : {y_pred[:10]}")  # Ajoutez ceci pour voir les prédictions
        return y_pred


In [ ]:
print(f"Exemple de X : {X[:5]}")
print(f"Exemple de y : {y[:5]}")


In [ ]:
# Transformer y en indices numériques
y_encoded = label_encoder.transform(y)
print(f"Exemple de y encodé : {y_encoded[:5]}")


In [ ]:
from collections import Counter

print(Counter(y))  # Distribution des maladies dans le dataset


In [ ]:
# Vérifiez les classes apprises par le LabelEncoder
print(label_encoder.classes_)


In [ ]:
# Vérifiez les 5 premières correspondances entre X et y
for symptoms, disease in zip(X[:5], y[:5]):
    print(f"Symptômes : {symptoms} => Maladie : {disease}")


In [ ]:
# Encoder y avec le LabelEncoder
y_encoded = label_encoder.transform(y)
print(f"Exemple de y encodé : {y_encoded[:5]}")
